# 为什么基础 RAG 还会答错

基础 RAG 能返回答案，只能说明整套流程可以运行，不能说明答案可靠。资料要经过解析、分块和检索，找到的内容还要交给模型生成回答。中间任何一步漏掉信息，结果都可能出错。

遇到答错、答不全或缺少资料依据的回答时，先确认错误发生在哪里。直接换模型或叠加新方法，往往会掩盖原来的问题。

## 先检查哪一步出了问题

拿出问题发生时的提问、检索结果和回答，依次检查：

1. 原文里有没有答案？
2. 解析和分块后，相关内容是否完整？
3. 检索结果里有没有正确内容？
4. 正确内容是否排得太靠后，以至于模型没有看到？
5. 模型是否按照找到的内容作答？

在哪一步发现问题，就先改那一步。比如，正确内容已经出现在检索结果中，继续调整向量模型通常不会解决问题。这时应当检查资料排序、交给模型的上下文或 Prompt（发给模型的指令）。

## 几种常见情况

| 现象 | 先检查 |
|---|---|
| 原文有答案，但系统没有找到 | 文档解析、分块和检索方式 |
| 找到了正确内容，但它排得很靠后 | 资料排序和筛选 |
| 找到的内容没有问题，回答却加入了原文没有的信息 | Prompt、引用方式和拒答规则 |
| 回答一个问题需要参考多处资料 | 上下文补充和多步检索 |
| 追问时忘记了上一轮内容 | 对话记录 |
| 不该检索时仍然检索，或者查错了资料 | 检索条件和资料范围 |

## 如何决定改资料、RAG 还是微调

先把“知识不在资料”“资料没有被找到”和“资料已经找到但回答方式不对”分开。下表中的“最低数据”只表示能够开始一次可复查的诊断，不是保证效果的样本数量。

| 改法 | 职责 | 最低数据 | 评估指标 | 不适用情况 |
|---|---|---|---|---|
| 改资料 / RAG 流程 | 修正资料来源、解析、分块、索引、过滤、检索、重排或上下文拼接，让正确证据可被找到并交给模型 | 原始资料（或明确的资料权限）和至少一条可复现的问题；要量化检索时再准备 query→evidence qrels | 资料覆盖率、证据 Recall@k、MRR、上下文完整性/精度，以及有据回答的正确性 | 正确证据已经在上下文而问题属于回答行为：不要继续调检索侧，先检查 Prompt 和回答约束；只有生成行为缺口稳定、重复且有可信对齐数据时，才酌情考虑生成器微调，并非必须微调。无法补充或授权资料时，不应靠 RAG 伪造答案，应明确拒答 |
| 微调检索器 | 调整 query 与 evidence 的向量/排序关系，只负责把相关证据排得更靠前，不负责写答案或补充事实 | 与真实任务一致的 query-positive evidence，最好还有经判断的难负例；按文档/章节/问题簇隔离 train、dev、test，并保留完整 qrels | Recall@1/3/5/10、MRR（必要时 nDCG）、逐题改善/退化和通用问题回归 | 解析、分块、索引或检索配置仍有明显错误，或正确证据已在上下文而问题出在生成；假负例很多也不宜直接训练 |
| 微调生成器 | 学习回答格式、语气、引用和拒答边界；在本教程的有据回答/RAG 契约中要求生成器只依据提供的上下文并可拒答。模型本身可能使用参数知识，但那不能当作可追溯证据；生成器微调也不能修复漏召回或替代动态资料 | 包含问题、实际上下文、目标回答的对齐样本，并标注正确性/有据性/格式或拒答行为；按任务和资料隔离验证集与测试集 | 回答正确性、faithfulness/groundedness、引用与格式遵循、拒答 precision/recall，以及通用能力回归 | 证据尚未稳定召回、事实经常更新，或没有可信的目标回答/拒答标签；这些情况先改资料/RAG 或 Prompt |

因此可按“资料覆盖 → 解析/分块/检索 → 检索器语义排序 → 生成行为”的顺序定位。RAG 的优势是能更新外部资料并保留来源；微调的优势是让检索或生成行为适应稳定任务，但两者都不能绕过数据质量和独立评估。

## 一条可复查的失败诊断 trace

下面用 canonical 数据包中的 `gaussian_mean_estimate` 做一条完整的失败 trace。检索器只接收 `query_id` 对应的问题文字和 `evidence.quote`；问题的 qrels 只在检索完成后用于复核，不能参与排序。为保证这页不假装调用回答模型，回答栏是固定的教学草稿，并明确标为“不是模型输出”。

这条 trace 不是统一 benchmark：它只展示怎样把一次“原文有答案但 top-k 没找全”的问题定位到检索/上下文阶段。

In [1]:
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start.resolve(), *start.resolve().parents):
        if (folder / "data" / "dataset/manifest.json").is_file() and (folder / "common" / "dataset.py").is_file():
            return folder
    raise FileNotFoundError("没有找到 C7 canonical 数据目录，请从教程目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))


In [2]:
from collections import defaultdict

from common.dataset import (
    load_search_evidence,
    load_qrel_records,
    load_query_only,
)
from common.eval_utils import (
    build_bm25_search,
    emit_tutorial_audit,
    normalize_text,
)

query_id = "gaussian_mean_estimate"
query_row = load_query_only([query_id])[0]
evidence_rows = load_search_evidence()

# 检索阶段只使用 canonical evidence 的 page/quote 投影，不读 qrels 或 reference_answer。
search_rows = [
    {"page": int(row["page"]), "text": str(row["quote"])}
    for row in evidence_rows
]
search = build_bm25_search(search_rows)
top_k = 4
retrieved = search(query_row["query"], top_k=top_k)
all_ranked = search(query_row["query"], top_k=len(search_rows))

ids_by_key = defaultdict(list)
for row in evidence_rows:
    ids_by_key[(int(row["page"]), normalize_text(row["quote"]))].append(row["evidence_id"])


def evidence_ids(item):
    return ids_by_key[(int(item.page), normalize_text(item.text))]


# 只有完成排序后才读取 qrels，作为事后诊断标签。
qrels = [
    row
    for row in load_qrel_records()
    if row["query_id"] == query_id and int(row.get("relevance", 0)) == 1
]
target_ids = {row["evidence_id"] for row in qrels}
current_evidence_ids = {row["evidence_id"] for row in evidence_rows}
assert target_ids <= current_evidence_ids
assert all(set(evidence_ids(item)) <= current_evidence_ids for item in retrieved)
target_rank = next(
    (rank for rank, item in enumerate(all_ranked, 1) if target_ids.intersection(evidence_ids(item))),
    None,
)
target_row = next(row for row in evidence_rows if row["evidence_id"] in target_ids)
context = "\n".join(normalize_text(item.text) for item in retrieved)
target_quote = normalize_text(target_row["quote"])
target_in_top_k = any(target_ids.intersection(evidence_ids(item)) for item in retrieved)
answer_draft = "当前 top-k 片段不足以核对 µc 的估计式；这里暂不提供参数估计结论。"
answer_supported = target_quote in context
source_has_answer = bool(target_ids)

if not source_has_answer:
    root_cause = "资料范围"
    next_action = "补充资料或明确拒答范围"
elif not target_in_top_k:
    root_cause = "检索/上下文"
    next_action = "先检查查询改写、检索方式和 top-k，再复跑同一问题"
elif not answer_supported:
    root_cause = "回答"
    next_action = "逐条核对回答结论与候选原文"
else:
    root_cause = "未发现明显问题"
    next_action = "保留该 trace，继续检查其他问题"

print("[query]", query_id, "→", query_row["query"])
print("[retriever] BM25 over canonical evidence.quote; top_k =", top_k)
for rank, item in enumerate(retrieved, 1):
    ids = evidence_ids(item)
    print(f"  {rank}. {ids} | 第 {item.page} 页 | score={item.score:.3f} | {normalize_text(item.text)[:100]}")
print("[answer draft]（固定教学草稿，不是模型输出）：", answer_draft)
print("[evidence check] 原文有答案：", source_has_answer, "；目标证据在 top-k：", target_in_top_k, "；回答草稿可由 top-k 直接核对：", answer_supported)
print("[post-hoc canonical evidence]", target_row["evidence_id"], "| 第", target_row["page"], "页 |", target_quote)
print("[rank] 目标证据完整排序位置：", target_rank)
print("[root cause]", root_cause, "；下一步：", next_action)

emit_tutorial_audit({
    "trace_kind": "c1_failure_diagnosis",
    "query_id": query_id,
    "retriever": "BM25 over canonical evidence.quote",
    "top_k": top_k,
    "retrieved_evidence_ids": [evidence_ids(item) for item in retrieved],
    "target_evidence_ids": sorted(target_ids),
    "target_rank": target_rank,
    "source_has_answer": source_has_answer,
    "target_in_top_k": target_in_top_k,
    "answer_supported_by_top_k": answer_supported,
    "root_cause": root_cause,
    "answer_origin": "fixed_teaching_draft_not_model_output",
})

assert source_has_answer and not target_in_top_k and not answer_supported
assert root_cause == "检索/上下文" and target_rank is not None


[query] gaussian_mean_estimate → 多元正态分布的 µc 参数如何估计？
[retriever] BM25 over canonical evidence.quote; top_k = 4
  1. ['evi_ft_chunk_e3ef1009b0d414d3'] | 第 79 页 | score=23.967 | θ1, P(C = c2) = θ2, ..., P(C = ck) = θk，那么显然C 服从参数为θ = (θ1, θ2, ..., θk) ∈Rk 的Categorical 分布，其概率质量函数
  2. ['evi_ft_chunk_2048c07a1e84fb7e'] | 第 28 页 | score=18.624 | 此时，C 只能取C 或者C + 1。若C 取C，则相当于升高了检验水平α；若C 取C + 1 则相当于降低了 检验水平α。具体如何取舍需要结合实际情况，一般的做法是使α 尽可能小，因此倾向于令C 取C
  3. ['evi_ft_chunk_e523fe0b60f92d62'] | 第 82 页 | score=16.800 | n) 其中t1, t2, ..., tn ∈[0, 1], Pn i=1 ti = 1。此不等式在概率论中通常以如下形式出现 φ(E[X]) ⩽E[φ(X)] 其中X 是随机变量，φ 为凸函数，E[X
  4. ['evi_ft_chunk_c1763611a3ee2c84'] | 第 14 页 | score=14.822 | 中将会讲述的线性回归、对数几率回归、决策树等。“算法”产出的结果称为“模型”， 通常是具体的函数或者可抽象地看作为函数，例如一元线性回归算法产出的模型即为形如f(x) = wx + b 的一元一次函数
[answer draft]（固定教学草稿，不是模型输出）： 当前 top-k 片段不足以核对 µc 的估计式；这里暂不提供参数估计结论。
[evidence check] 原文有答案： True ；目标证据在 top-k： False ；回答草稿可由 top-k 直接核对： False
[post-hoc canonical evidence] evi_3e26c517329f | 第 77 页 | ˆµc = ¯x = 1 n n X i=1 

同一种现象可能有不同原因。修改前先保存当时的提问、检索结果和回答，修改后再用同一个问题比较。否则只能看到答案变了，无法确认问题是否解决。

例如，问题“第 2.2 节列出的三种模型评估办法分别叫什么？”可以直接进入[给片段补充所属上下文](../3. 索引阶段/给片段补充所属上下文.ipynb)查看：先检查第 18 页是否被检索到，再比较补充背景前后的排名。读完后可以回到[教程首页](../README.md)，按遇到的问题选择对应章节；需要整理提问、检索结果和回答时，再进入第 7 章的评估准备页。

## 本页导航

- 章节入口：[本章 README](README.md)
- 运行准备：[C7 统一运行准备](../README.md#运行准备)
- 相关下一步：[分块优化](../2.%20数据处理/分块优化.ipynb)

